# Kingdom Identification via Protein Embeddings

Welcome to the RNA Museum! At this museum, we host records of all known RNA sequences, along with information about their origin, structure, and function. Recently, there was a huge mix-up at the lab, where *Eukaryotic*, *Prokaryotic*, and *Viral* RNA sequences were all combined into a single solution! In the first part of our isolation protocol, we want to figure out which sequences belong to which kingdom.

## Learning objectives
- Understand transfer learning with embeddings
- Generate embeddings from pretrained models
- Implement multiclass classifier in `sklearn` and `PyTorch`
- Design and run comparative experiments on protein-related prediction tasks
- Analyze and interpret your results and draw conclusions based on a test set

### Library Imports

We provide the most likely imports, but you **WILL** need to add more as you go along.

In [ ]:
# !pip install -q pandas numpy matplotlib seaborn scikit-learn torch

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Part 1: The Biological Problem

Proteins from different kingdoms of life — Eukaryotes, Prokaryotes, and Viruses — have evolved under different selective pressures. As a result, their sequences carry subtle signatures of their origin. Our goal is to build a classifier that can identify which kingdom a protein belongs to, using only its sequence (represented as a numerical embedding).

This is a **3-class classification** problem with the classes:
1. **Eukaryota** — organisms with complex cells (e.g., animals, plants, fungi)
2. **Prokaryota** — organisms with simple cells (e.g., bacteria, archaea)
3. **Viruses** — obligate intracellular parasites

## Part 2: Reading in Protein Embeddings

Instead of running the model on 10,000 sequences, we will instead read in pre-computed embeddings that were generated using ESM-2 on a computing cluster.

Models like ESM-2 are massive neural networks trained on millions of known protein sequences. By learning the "language" of proteins, they can convert any amino acid sequence into a rich numerical vector.

In [ ]:
embeddings = np.load('data/protein_kingdom/protein_embeddings.npy')
df = pd.read_csv('data/protein_kingdom/uniprot_10000_proteins.csv')

print("Shape of embeddings matrix:", embeddings.shape)
print(f"This means we have {embeddings.shape[0]} proteins, each represented by a vector of {embeddings.shape[1]} numbers.")

In [ ]:
df.sample(n=10, random_state=42)

## Part 3: Visualizing the Protein Embeddings

We can use dimensionality reduction techniques like PCA or t-SNE to visualize the high-dimensional embeddings in 2D or 3D space. See **HMB301 Module 6** if you need a refresher on these techniques.


---
**GQ*1 (2 pts): Use a dimensionality reduction technique to visualize the protein embeddings. Colour the points by their corresponding kingdom. Do you see any clusters forming? What's the general pattern of the data?**
> Hint: Remember to use `StandardScaler` to scale the data before PCA-based methods.

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# TODO

<span style="background-color: #FFD700">**Write your answer below**</span>

Answer here:

---

## Part 4: Data Prepping

To train our models, we will need to split them into training, validation, and test sets. Additionally, we will need to encode the labels into numerical formats. Run the code cells below.

In [ ]:
from sklearn.preprocessing import LabelEncoder

X = embeddings

# Encode kingdom labels as integers
le = LabelEncoder()
y = le.fit_transform(df['kingdom'])

# Store the class names for later use
class_names = list(le.classes_)
NUM_CLASSES = len(class_names)

print(f"Classes: {class_names}")
print(f"Number of classes: {NUM_CLASSES}")
print(f"Label shape: {y.shape}")
print(f"Example labels: {y[:10]}")

In [ ]:
from sklearn.model_selection import train_test_split

# First split: 80% train+valid, 20% test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: of the 80%, take 75% train and 25% valid (i.e., 60/20/20 overall)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

In [ ]:
# Verify split sizes
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_valid.shape, y_valid.shape)
print("Test set shape:", X_test.shape, y_test.shape)

Now, we have a training, validation, and test set of protein embeddings. We will first try a variety of classical machine learning models using `sklearn`. Then, we will try to use clustering. Finally, we will implement a simple feedforward neural network using `PyTorch` to see if we can improve our performance.

---
**GQ*2 (6 pts): Choose 2 of the following models to implement. Perform *some* hyperparameter tuning on the validation set to find the best hyperparameters for each model. Do not evaluate on the test set! (We will evaluate the performance of the models in GQ4.) Discuss which of the two models you implemented performs better, and explain why. For each model, discuss why a certain hyperparameter choice performs better than others.**

- **k-Nearest Neighbors**
  > Hint: try different `n_neighbors` values! You'll also want to reduce the dimensionality of the data before applying kNN, as it can struggle in high dimensions.
- **Logistic Regression**
   > Hint: only tune the `C` hyperparameter!
- **Random Forests**
   > Hint: try different `n_estimators` and `max_depth` values!
- **XGBoost**
   > Hint: try different `n_estimators` and `learning_rate` values!

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# TODO:

<span style="background-color: #FFD700">**Write your answer below**</span>

Answer here:

---

## Part 5: Using Neural Networks

Now that you've established a baseline with classical machine learning models, let's see if we can do better with a simple feedforward neural network implemented using `PyTorch`. You can use the same training and validation sets that you created for the classical models.

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

Now, we'll create an Embedding `Dataset` and `DataLoader` to feed our data into the neural network.

In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        # Convert numpy arrays to PyTorch tensors
        self.embeddings = torch.tensor(embeddings)
        self.labels = torch.tensor(labels)

    def __len__(self):
        # Return length of dataset in number of samples
        return len(self.labels)

    def __getitem__(self, idx):
        # Return slice of a dataset to be used as a batch
        return self.embeddings[idx], self.labels[idx]

In [ ]:
# Create Datasets
train_dataset = EmbeddingDataset(X_train, y_train)
valid_dataset = EmbeddingDataset(X_valid, y_valid)
test_dataset = EmbeddingDataset(X_test, y_test)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

The following code is the `FlexibleFullyConnectedNetwork` from Module 5. We will use this same architecture here, with the same ability for you to specify the number of layers and their width.

In [ ]:
# Define dimensions
INPUT_DIM = X_train.shape[1]
NUM_CLASSES = 3

class FlexibleFullyConnectedNetwork(nn.Module):
    def __init__(self, layer_dims, activation=nn.ReLU):
        super(FlexibleFullyConnectedNetwork, self).__init__()

        # Creating the layers using ModuleList
        layers = nn.ModuleList()

        # Hidden layers
        for i in range(len(layer_dims) - 2):
            layers.append(nn.Linear(layer_dims[i], layer_dims[i + 1]))  # Fully connected layer
            layers.append(activation())                                 # Activation function

        # Output layer
        layers.append(nn.Linear(layer_dims[-2], layer_dims[-1]))        # Output layer

        # Store the layers in the class
        self.layers = layers

    def forward(self, x):
        # Pass x through all layers
        for layer in self.layers:
            x = layer(x)

        return x

---
**GQ*3 (10 pts): Implement the training loop for the neural network.**

**Here are a few guidelines:**

1. **For your loss function, use `nn.CrossEntropyLoss()`, which is appropriate for multi-class classification problems.**
2. **For your optimizer, you can use `torch.optim.Adam`.**
3. **Train the model for a reasonable number of epochs (e.g., 10-15).**
4. **Plot loss and accuracy curves for the training and validation sets across epochs.**
5. **Tune the *number of layers* and the *width of the layers*, trying at least 3 different values for each.**

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# TODO

---
## Part 6: Comparing Models Head to Head

Now that you've trained both classical and deep learning models, it's time to compare them head-to-head on the test set, which we've been holding out on this entire time.

---
**GQ*4 (10 pts): Of the two classical models you trained and the neural network, show which one performed best on the test set.**

**Here are some guidelines for your analysis:**
1. **Ensure that you re-train your models only on the training set (not the validation set) with the optimal hyperparameters from the test set.**
2. **If you performed steps like PCA-transforming the data, ensure you only `fit_transform` on the training set, and then `transform` on the test set.**
3. **Report the test set accuracy for each model, and show a confusion matrix for the best-performing model.**
4. **Plot the confusion matrix for the test set on both the best performing of the classic models and the neural network model, and compare them. Where does the model make the most mistakes? Are there certain kingdoms that are more likely to be confused with each other?**
5. **Discuss the results. Discuss the biological implications of your model. Why do you think a certain model performed better than the others?**

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# TODO

<span style="background-color: #FFD700">**Write your answer below**</span>

Answer here:

---

## Conclusion

Congratulations on completing the capstone project! You have successfully navigated a full machine learning pipeline in bioinformatics. You started with raw biological data, leveraged powerful pretrained embeddings, explored the data visually, established performance baselines with classical models, and engineered your own neural network.

The process of experimenting, logging results, and interpreting performance is the core daily work of a machine learning scientist. The skills you've consolidated here are fundamental and will serve you well in any future data science endeavour.